# DoshaNet Data Prep

This notebook prepares data and writes outputs to a new run folder so existing artifacts are not overwritten.

In [1]:
import os, json, pickle
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
RUN_DIR = os.path.join('runs', f"exp_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(RUN_DIR, exist_ok=True)
print('RUN_DIR:', RUN_DIR)

RUN_DIR: runs/exp_20260429_132506


In [3]:
with open('prakriti_clean.json', 'r') as f:
    df = pd.DataFrame(json.load(f))

feature_cols = [c for c in df.columns if c != 'Dosha']
dosha_names = sorted(df['Dosha'].unique())

encoders = {}
for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

enc_path = os.path.join(RUN_DIR, 'encoders.pkl')
with open(enc_path, 'wb') as f:
    pickle.dump(encoders, f)
print('Saved:', enc_path)

Saved: runs/exp_20260429_132506/encoders.pkl


In [4]:
X = df[feature_cols].values.astype(np.float32)
y = df['Dosha'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_full = np.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])

train_mask = np.zeros(len(X_full), dtype=bool)
test_mask = np.zeros(len(X_full), dtype=bool)
train_mask[:len(X_train)] = True
test_mask[len(X_train):] = True

print('Total patients:', len(X_full), '| Features:', len(feature_cols))
print('Train:', len(X_train), '| Test:', len(X_test))

Total patients: 1199 | Features: 29
Train: 959 | Test: 240


In [5]:
prep_path = os.path.join(RUN_DIR, 'prep_data.npz')
np.savez_compressed(
    prep_path,
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    X_full=X_full, y_full=y_full,
    feature_cols=np.array(feature_cols, dtype=object),
    dosha_names=np.array(dosha_names, dtype=object),
    train_mask=train_mask, test_mask=test_mask
)
print('Saved:', prep_path)
print('Use RUN_DIR in train.ipynb:', RUN_DIR)

Saved: runs/exp_20260429_132506/prep_data.npz
Use RUN_DIR in train.ipynb: runs/exp_20260429_132506
